In [1]:
!pip install wandb torchsummary thop --quiet


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
import wandb
from thop import profile


In [3]:
wandb.login()
wandb.init(
    project="cifar10-cnn-gradient-analysis-Lab2",
    name="CustomCNN-CIFAR10"
)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: abhikashyap4563 (abhikashyap4563-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
class CustomCIFAR10(Dataset):
    def __init__(self, train=True):
        self.transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.4914, 0.4822, 0.4465],
                std=[0.2023, 0.1994, 0.2010]
            )
        ])
        self.dataset = datasets.CIFAR10(
            root="./data",
            train=train,
            download=True,
            transform=self.transform
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        return image, label


In [5]:
train_loader = DataLoader(
    CustomCIFAR10(train=True),
    batch_size=128,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    CustomCIFAR10(train=False),
    batch_size=128,
    shuffle=False,
    num_workers=2
)


100%|██████████| 170M/170M [00:05<00:00, 31.4MB/s]


In [6]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)


In [8]:
dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(model, inputs=(dummy_input,), verbose=False)

print(f"FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Params: {params/1e6:.2f} M")

wandb.log({
    "FLOPs (MFLOPs)": flops / 1e6,
    "Parameters (M)": params / 1e6
})


FLOPs: 42.08 MFLOPs
Params: 2.47 M


In [9]:
def plot_gradient_flow(model):
    ave_grads = []
    layers = []

    for name, param in model.named_parameters():
        if param.requires_grad and "bias" not in name:
            layers.append(name)
            ave_grads.append(param.grad.abs().mean().cpu())

    plt.figure(figsize=(12,5))
    plt.plot(ave_grads)
    plt.xticks(range(len(layers)), layers, rotation="vertical")
    plt.ylabel("Average Gradient")
    plt.title("Gradient Flow")
    plt.grid(True)
    return plt


In [10]:
def plot_weight_updates(model, prev_weights):
    deltas = []
    layers = []

    for (name, param), prev in zip(model.named_parameters(), prev_weights):
        if param.requires_grad and "bias" not in name:
            layers.append(name)
            deltas.append((param.data - prev).abs().mean().cpu())

    plt.figure(figsize=(12,5))
    plt.plot(deltas)
    plt.xticks(range(len(layers)), layers, rotation="vertical")
    plt.ylabel("Average Weight Update")
    plt.title("Weight Update Flow")
    plt.grid(True)
    return plt


In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    prev_weights = [p.clone().detach() for p in model.parameters()]

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    acc = 100 * correct / total

    grad_fig = plot_gradient_flow(model)
    wandb.log({"Gradient Flow": wandb.Image(grad_fig)})
    plt.close()

    weight_fig = plot_weight_updates(model, prev_weights)
    wandb.log({"Weight Update Flow": wandb.Image(weight_fig)})
    plt.close()

    wandb.log({
        "Epoch": epoch + 1,
        "Train Loss": running_loss / len(train_loader),
        "Train Accuracy": acc
    })

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f}, Acc: {acc:.2f}%")


Epoch [1/30] Loss: 661.3761, Acc: 38.07%
Epoch [2/30] Loss: 511.0952, Acc: 52.40%
Epoch [3/30] Loss: 447.9679, Acc: 59.14%
Epoch [4/30] Loss: 409.1200, Acc: 63.15%
Epoch [5/30] Loss: 382.4979, Acc: 65.56%
Epoch [6/30] Loss: 359.7930, Acc: 67.77%
Epoch [7/30] Loss: 343.8033, Acc: 69.21%
Epoch [8/30] Loss: 328.3336, Acc: 70.92%
Epoch [9/30] Loss: 313.7964, Acc: 72.31%
Epoch [10/30] Loss: 302.3286, Acc: 73.44%
Epoch [11/30] Loss: 290.0716, Acc: 74.38%
Epoch [12/30] Loss: 281.7750, Acc: 75.21%
Epoch [13/30] Loss: 271.2037, Acc: 76.42%
Epoch [14/30] Loss: 259.7626, Acc: 77.21%
Epoch [15/30] Loss: 251.5035, Acc: 78.24%
Epoch [16/30] Loss: 245.1030, Acc: 78.67%
Epoch [17/30] Loss: 236.8410, Acc: 79.43%
Epoch [18/30] Loss: 232.0923, Acc: 79.83%
Epoch [19/30] Loss: 224.8999, Acc: 80.64%
Epoch [20/30] Loss: 218.2859, Acc: 81.12%
Epoch [21/30] Loss: 212.7249, Acc: 81.83%
Epoch [22/30] Loss: 206.8386, Acc: 82.10%
Epoch [23/30] Loss: 201.0004, Acc: 82.48%
Epoch [24/30] Loss: 195.1529, Acc: 83.09%
E

In [12]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_acc = 100 * correct / total
wandb.log({"Test Accuracy": test_acc})
print(f"Test Accuracy: {test_acc:.2f}%")


Test Accuracy: 83.32%
